# 3.1 — MobileViT fine-tuning completo local — Food-101

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoslund/ViT-for-101-food-app/blob/main/notebooks/3.1-mobilevit.ipynb)

Entrenamiento completo de **MobileViT** usando la estructura del proyecto y los artefactos generados por `2.0-preprocessing.ipynb`.

- Usa `config.py` y `preprocessing/loaders.py`.
- Usa los **68.175 ejemplos de train** y **7.575 de validation** ya definidos.
- Reserva el `test` oficial para la evaluación final, y reporta aparte el **subset del benchmark por tercil de dificultad**, que es lo que contesta la pregunta del proyecto.
- Mide el costo arquitectónico (parámetros, FLOPs, latencia en GPU y CPU) con `modeling/evaluation.py`, el mismo código que usa ViT.
- Guarda predicciones y métricas en `reports/results/mobilevit/` para compararlas con ViT sin reentrenar.
- Localmente no hace `git clone`, no instala paquetes, no descarga datos y no crea splits nuevos. En Colab sí: ver §0.
- Guarda checkpoints bajo `models/mobilevit/full/`.

## 0. Antes de ejecutar

Desde la raíz del repo:

```bash
uv sync --extra deep
uv run jupyter lab
```

El preprocessing debe existir previamente, por ejemplo con:

```bash
make preprocess
```

o ejecutando `notebooks/2.0-preprocessing.ipynb`.

**Windows:** el `torch` de PyPI solo usa CPU. Para usar la GPU, después del `uv sync`:

```bash
uv pip install --reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu128
```

y no volver a correr `uv sync` (reinstala el torch de CPU). Abrir Jupyter con el entorno activado (`.venv\Scripts\activate`) en lugar de `uv run`.

Conviene correr antes `3.0-mobilevit-smoke-test.ipynb`: usa la misma receta y el mismo protocolo de evaluación en ~15 minutos.

## 0.1 Parámetros de ejecución

- `USE_DRIVE` (solo Colab): monta Google Drive y guarda ahí el `.tar.gz` del dataset, los checkpoints y los resultados. El disco de Colab se borra al desconectarse; sin Drive, una desconexión a mitad de entrenamiento pierde todo.

- `RESUME`: si la corrida se cortó (desconexión de Colab, PC suspendida), `True` retoma desde el último checkpoint de `OUTPUT_DIR` en lugar de empezar de cero. Con `False` y checkpoints viejos en la carpeta, la notebook se detiene en vez de mezclarlos.

Localmente estos parámetros no cambian nada salvo `RESUME`: todo se lee y escribe en las rutas de `config.py`.

In [1]:
# Solo Colab: persistir dataset, checkpoints y resultados en Google Drive.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/ceia-vpc3"

# True = retomar desde el ultimo checkpoint de OUTPUT_DIR (corrida cortada).
RESUME = False

## 0.2 Entorno (Colab)

Mismo patrón que `2.0-preprocessing.ipynb`: en Colab clona el repo e instala el paquete con el extra `deep`. `torch` ya viene con CUDA en Colab, así que `pip` no lo reinstala. Localmente esta celda solo verifica que el paquete esté instalado.

In [2]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Colab: {IN_COLAB}")


def _paquete_disponible():
    try:
        import vit_for_101_food_app  # noqa: F401

        return True
    except ImportError:
        return False


if IN_COLAB:
    REPO_URL = "https://github.com/marcoslund/ViT-for-101-food-app.git"
    REPO_DIR = "/content/ViT-for-101-food-app"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)

    if not _paquete_disponible():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[deep]"], check=True
        )
    os.chdir(f"{REPO_DIR}/notebooks")

    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
elif not _paquete_disponible():
    raise RuntimeError(
        "vit_for_101_food_app no esta instalado en este kernel. "
        "Elegí el kernel del .venv del proyecto (ver 'Antes de ejecutar')."
    )

print("paquete disponible:", _paquete_disponible())

2026-09-24 15:56:13.953 | INFO     | vit_for_101_food_app.config:<module>:11 - PROJ_ROOT path is: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app


Colab: False
paquete disponible: True


## 0.3 Datos (Colab)

En Colab el disco arranca vacío en cada sesión, así que acá se hace lo que localmente hace `make preprocess`: descarga (o copia desde Drive) el dataset, genera el split si no está versionado en el repo y arma el cache. El split es determinístico (semilla fija), así que regenerarlo da el mismo `sha256`.

Localmente esta celda no hace nada: los datos tienen que existir de antes.

In [3]:
if IN_COLAB:
    import shutil

    from vit_for_101_food_app import config
    from vit_for_101_food_app import dataset as prep

    tar_local = config.RAW_DATA_DIR / "food-101.tar.gz"
    tar_drive = f"{DRIVE_DIR}/food-101.tar.gz"
    config.RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if USE_DRIVE and os.path.exists(tar_drive) and not tar_local.exists():
        print("copiando el dataset desde Drive...")
        shutil.copy(tar_drive, tar_local)

    prep.download()  # descarga (~5 GB) y extrae; idempotente
    if USE_DRIVE and not os.path.exists(tar_drive):
        print("guardando el dataset en Drive para la proxima sesion...")
        shutil.copy(tar_local, tar_drive)

    if not config.TRAIN_VAL_SPLIT.exists():
        prep.split()
    prep.cache(workers=os.cpu_count() or 2)  # idempotente

In [4]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForImageClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score

try:
    from vit_for_101_food_app import config
    from vit_for_101_food_app.modeling import evaluation
    from vit_for_101_food_app.preprocessing import cache, loaders, splits
except ImportError as exc:
    raise RuntimeError(
        "No se pudo importar el paquete del proyecto. "
        "Ejecutá `uv sync --extra deep` y abrí Jupyter con `uv run jupyter lab`."
    ) from exc

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("Dispositivo: CPU")
    print("AVISO: el full training en CPU puede tardar muchas horas.")

c:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.16
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.0 GB


## 1. Verificar artefactos de preprocessing

In [5]:
required_files = [
    config.TRAIN_VAL_SPLIT,
    config.TRAIN_VAL_MANIFEST,
    config.LABEL_MAP,
    config.CACHE_DIR / "cache_manifest.json",
]

missing = [Path(p) for p in required_files if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos del preprocessing:\n"
        + "\n".join(f" - {p}" for p in missing)
        + "\n\nEjecutá primero `make preprocess`."
    )

if not cache.cache_is_valid(
    cache_dir=config.CACHE_DIR,
    short_side=config.CACHE_SHORT_SIDE,
):
    raise RuntimeError("El cache existe pero no es válido. Regeneralo con `make cache`.")

id2label, label2id = splits.load_label_map(config.LABEL_MAP)
NUM_LABELS = len(label2id)

print("Preprocessing OK")
print("train/val split:", config.TRAIN_VAL_SPLIT)
print("label map:", config.LABEL_MAP)
print("cache:", config.CACHE_DIR)
print("clases:", NUM_LABELS)

assert NUM_LABELS == 101

Preprocessing OK
train/val split: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\data\processed\train_val_split.csv
label map: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\data\processed\label_map.json
cache: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\data\interim\food-101-288
clases: 101


## 2. DataLoaders completos de MobileViT

Se reutiliza `loaders.build_dataloaders("mobilevit", ...)`; no se reimplementan resize, crop, BGR/RGB, escala ni augmentation.

In [6]:
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    BATCH_SIZE = 32 if gpu_mem_gb >= 12 else 16
else:
    BATCH_SIZE = 8

NUM_WORKERS = min(4, os.cpu_count() or 1)  # Colab tiene 2 CPUs

dls = loaders.build_dataloaders(
    "mobilevit",
    train_policy="standard",
    source="cache",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    splits_to_load=("train", "val", "test"),
    images_root=config.CACHE_DIR,
    csv_path=config.TRAIN_VAL_SPLIT,
    label_map_path=config.LABEL_MAP,
    meta_dir=config.FOOD101_META_DIR,
)

train_ds = dls["train"].dataset
val_ds = dls["val"].dataset
test_ds = dls["test"].dataset

print("batch size:", BATCH_SIZE)
print("train:", len(train_ds))
print("validation:", len(val_ds))
print("test:", len(test_ds))

assert len(train_ds) == 68175
assert len(val_ds) == 7575
assert len(test_ds) == 25250

batch = next(iter(dls["train"]))
print("pixel_values:", tuple(batch["pixel_values"].shape), batch["pixel_values"].dtype)
print("labels:", tuple(batch["labels"].shape))

batch size: 16
train: 68175
validation: 7575
test: 25250
pixel_values: (16, 3, 256, 256) torch.float32
labels: (16,)


## 3. MobileViT preentrenado

In [7]:
CHECKPOINT = config.MODELS["mobilevit"]
print("checkpoint:", CHECKPOINT)

model = AutoModelForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

params_total = sum(p.numel() for p in model.parameters())
params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"parámetros totales: {params_total / 1e6:.2f} M")
print(f"parámetros entrenables: {params_trainable / 1e6:.2f} M")

checkpoint: apple/mobilevit-small


[transformers] You passed `num_labels=101` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 347/347 [00:00<00:00, 13300.41it/s]
[transformers] MobileViTForImageClassification LOAD REPORT from: apple/mobilevit-small
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 640]) vs model:torch.Size([101, 640])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([101])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


parámetros totales: 5.00 M
parámetros entrenables: 5.00 M


## 4. Métricas

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

## 5. Configuración del fine-tuning

Misma receta que el smoke test (`3.0`):

- Hasta **20 épocas**, con early stopping de paciencia 3: 20 es un techo, no una obligación.
- El mejor checkpoint se elige por **F1 macro** en validation.
- **Warmup lineal** durante el primer 5 % de los pasos.
- `LEARNING_RATE = 5e-4` es alto para fine-tuning (lo habitual ronda `5e-5`) y **no está elegido con evidencia**. Para que la comparación sea justa, ViT tiene que usar el mismo criterio de elección de hiperparámetros.

In [9]:
OUTPUT_DIR = config.MODELS_DIR / "mobilevit" / "full"
if IN_COLAB and USE_DRIVE:
    OUTPUT_DIR = Path(DRIVE_DIR) / "models" / "mobilevit" / "full"

EPOCHS = 20
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP = 0.05  # fracción de los pasos totales
EARLY_STOPPING_PATIENCE = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints_previos = sorted(OUTPUT_DIR.glob("checkpoint-*"))
if checkpoints_previos and not RESUME:
    raise RuntimeError(
        f"{OUTPUT_DIR} ya tiene checkpoints ({[p.name for p in checkpoints_previos]}). "
        "Si la corrida anterior se corto y queres seguirla, poné RESUME = True en §0.1; "
        "si queres empezar de cero, borrá esa carpeta."
    )

RESULTS_DIR = config.REPORTS_DIR / "results" / "mobilevit"
if IN_COLAB and USE_DRIVE:
    RESULTS_DIR = Path(DRIVE_DIR) / "results" / "mobilevit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("output:", OUTPUT_DIR)
print("resultados:", RESULTS_DIR)
print("epochs máx.:", EPOCHS)
print("learning rate:", LEARNING_RATE)
print("warmup:", WARMUP)
print("early stopping patience:", EARLY_STOPPING_PATIENCE)
print("best model metric: f1_macro")

output: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\models\mobilevit\full
resultados: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\reports\results\mobilevit
epochs máx.: 20
learning rate: 0.0005
warmup: 0.05
early stopping patience: 3
best model metric: f1_macro


In [10]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=NUM_WORKERS,
    report_to="none",
    seed=int(config.SEED),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

## 6. Fine-tuning completo

In [11]:
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=RESUME or None)
elapsed_train = time.time() - t0

print(f"\nTiempo total de entrenamiento: {elapsed_train / 60:.1f} min = {elapsed_train / 3600:.2f} h")
print("best checkpoint:", trainer.state.best_model_checkpoint)
print("best F1 macro:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.683174,1.485801,0.613201,0.605181,0.605181
2,1.253905,1.155322,0.695710,0.696675,0.696675
3,1.016730,1.075412,0.722772,0.723140,0.723140
4,0.877882,1.001348,0.742706,0.738835,0.738835
5,0.830868,1.050747,0.742178,0.742979,0.742979
6,0.738310,0.926387,0.767921,0.768556,0.768556
7,0.708863,0.895109,0.781386,0.780295,0.780295
8,0.613759,0.895821,0.785215,0.786753,0.786753
9,0.557466,0.869239,0.793003,0.792334,0.792334
10,0.481039,0.871240,0.800792,0.799971,0.799971


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 33.89it/s]



Tiempo total de entrenamiento: 114.8 min = 1.91 h
best checkpoint: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\models\mobilevit\full\checkpoint-85220
best F1 macro: 0.8268775912915304


## 7. Historial por época

In [12]:
history = pd.DataFrame(trainer.state.log_history)

epoch_metrics = history[
    history["eval_loss"].notna()
][
    ["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro", "eval_f1_weighted"]
].copy()

epoch_metrics

,epoch,eval_loss,eval_accuracy,eval_f1_macro,eval_f1_weighted
21,1.0,1.485801,0.613201,0.605181,0.605181
43,2.0,1.155322,0.695710,0.696675,0.696675
65,3.0,1.075412,0.722772,0.723140,0.723140
88,4.0,1.001348,0.742706,0.738835,0.738835
110,5.0,1.050747,0.742178,0.742979,0.742979
132,6.0,0.926387,0.767921,0.768556,0.768556
155,7.0,0.895109,0.781386,0.780295,0.780295
177,8.0,0.895821,0.785215,0.786753,0.786753
199,9.0,0.869239,0.793003,0.792334,0.792334
222,10.0,0.871240,0.800792,0.799971,0.799971


## 8. Guardar el mejor modelo

In [13]:
BEST_DIR = OUTPUT_DIR / "best"
trainer.save_model(str(BEST_DIR))
print("Mejor modelo guardado en:", BEST_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 31.73it/s]

Mejor modelo guardado en: C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\models\mobilevit\full\best


## 9. Evaluación final sobre test

`test` se usa recién al final, después de seleccionar el modelo con validation, y una sola vez.

Las predicciones se guardan **por imagen** (`predictions_test.csv`): con ese archivo se recalcula cualquier métrica, y se compara contra ViT imagen por imagen, sin volver a correr el modelo.

In [14]:
t0 = time.time()
test_output = trainer.predict(test_ds)
elapsed_test = time.time() - t0

test_frame = splits.load_split("test", csv_path=config.TRAIN_VAL_SPLIT, meta_dir=config.FOOD101_META_DIR)
test_preds = evaluation.predictions_frame(test_frame, test_output.predictions, id2label)
test_preds.to_csv(RESULTS_DIR / "predictions_test.csv", index=False)

test_metrics = evaluation.split_metrics(test_preds)
print(f"Tiempo test: {elapsed_test:.1f} s")
pd.Series(test_metrics)

Tiempo test: 57.9 s


n                25250.000000
n_clases           101.000000
accuracy             0.870455
top5_accuracy        0.970297
f1_macro             0.870131
f1_weighted          0.870131
dtype: float64

## 10. Subset del benchmark por tercil de dificultad

Esta es la tabla que contesta la pregunta del proyecto. El subset (`data/processed/benchmark_subset.csv`, 25 imágenes × 101 clases) es **el mismo archivo para todos los modelos**, y cada clase tiene un tercil de dificultad medido en el EDA (sección 5).

Si la brecha entre ViT y MobileViT es plana a lo largo de los terciles, el modelo en dispositivo alcanza; si ViT gana sobre todo en `dificil`, ahí está lo que paga la API.

`load_benchmark_subset` verifica el `sha256` del CSV contra su manifiesto antes de usarlo.

In [15]:
subset = evaluation.load_benchmark_subset()
benchmark = evaluation.benchmark_metrics(test_preds, subset)
benchmark.to_csv(RESULTS_DIR / "benchmark_por_tercil.csv")
benchmark

,n,n_clases,accuracy,top5_accuracy,f1_macro,f1_weighted
grupo,,,,,,
subset,2525,101,0.880396,0.966337,0.879907,0.879907
facil,850,34,0.937647,0.983529,0.957614,0.957614
medio,825,33,0.894545,0.974545,0.925768,0.925768
dificil,850,34,0.809412,0.941176,0.839653,0.839653


## 11. Reporte por clase

Ordenado por F1: las clases peores y mejores son las que interesa mirar, no las primeras del alfabeto. El reporte completo queda en `report_por_clase_test.csv`.

In [16]:
class_names = [id2label[i] for i in range(NUM_LABELS)]

report = classification_report(
    test_preds["label_id"],
    test_preds["pred_id"],
    labels=list(range(NUM_LABELS)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T.loc[class_names].sort_values("f1-score")
report_df.to_csv(RESULTS_DIR / "report_por_clase_test.csv")

print("10 clases con peor F1")
display(report_df.head(10))
print("10 clases con mejor F1")
display(report_df.tail(10))

10 clases con peor F1


,precision,recall,f1-score,support
steak,0.598394,0.596,0.597194,250.0
chocolate_mousse,0.727660,0.684,0.705155,250.0
filet_mignon,0.718876,0.716,0.717435,250.0
bread_pudding,0.726531,0.712,0.719192,250.0
pork_chop,0.737500,0.708,0.722449,250.0
apple_pie,0.764444,0.688,0.724211,250.0
foie_gras,0.736000,0.736,0.736000,250.0
huevos_rancheros,0.835821,0.672,0.745011,250.0
ravioli,0.750000,0.744,0.746988,250.0
cheesecake,0.729323,0.776,0.751938,250.0


10 clases con mejor F1


,precision,recall,f1-score,support
french_fries,0.937500,0.960,0.948617,250.0
macarons,0.944882,0.960,0.952381,250.0
bibimbap,0.971074,0.940,0.955285,250.0
spaghetti_carbonara,0.956175,0.960,0.958084,250.0
oysters,0.956175,0.960,0.958084,250.0
spaghetti_bolognese,0.963710,0.956,0.959839,250.0
lobster_roll_sandwich,0.971429,0.952,0.961616,250.0
seaweed_salad,0.952941,0.972,0.962376,250.0
deviled_eggs,0.960474,0.972,0.966203,250.0
edamame,0.980392,1.000,0.990099,250.0


## 12. Confusiones más frecuentes

Una matriz de 101 × 101 no se lee; los pares (clase real → clase predicha) con más errores, sí. Se pueden contrastar con el vecino más confundible de cada clase según el EDA (`class_difficulty.csv`).

In [17]:
errores = test_preds[~test_preds["correct"]]
confusiones = (
    errores.groupby(["class_dir", "pred_class"]).size()
    .rename("n").sort_values(ascending=False).reset_index()
)
confusiones.to_csv(RESULTS_DIR / "confusiones_test.csv", index=False)

print("errores totales:", len(errores), "de", len(test_preds))
confusiones.head(15)

errores totales: 3271 de 25250


,class_dir,pred_class,n
0,steak,filet_mignon,41
1,chocolate_mousse,chocolate_cake,32
2,filet_mignon,steak,31
3,steak,prime_rib,24
4,beef_tartare,tuna_tartare,21
5,club_sandwich,grilled_cheese_sandwich,21
6,chocolate_cake,chocolate_mousse,20
7,prime_rib,steak,19
8,pork_chop,steak,19
9,gyoza,dumplings,17


## 13. Costo arquitectónico: parámetros, FLOPs y latencia

Se mide con `modeling/evaluation.py`, el mismo código que usa ViT: esa es la condición para que la comparación sea entre arquitecturas.

- **FLOPs** por imagen a la resolución nativa del modelo. Se reportan también MACs (= FLOPs / 2), porque el paper de MobileViT llama "FLOPs" a los MACs.
- **Latencia** con batch 1 (una foto por vez, el caso de la app), mediana y p90 de 100 corridas, en **GPU y en CPU**. La de CPU es la más cercana al régimen en dispositivo; ningún número de acá es la latencia de un teléfono.

Para que la latencia sea comparable con la de ViT: mismo equipo, enchufado, y sin otros procesos pesados.

In [18]:
device = trainer.args.device
sample = test_ds[0]["pixel_values"].unsqueeze(0)

n_params = sum(p.numel() for p in model.parameters())
costo = {
    "params_m": n_params / 1e6,
    "size_mb_fp32": n_params * 4 / 1024**2,
    **evaluation.count_flops(model, sample.to(device)),
}
latencia = {
    "gpu": evaluation.measure_latency(model, sample, device) if device.type == "cuda" else None,
    "cpu": evaluation.measure_latency(model, sample, "cpu"),
}
model.to(device)

print(pd.Series(costo))
pd.DataFrame({k: v for k, v in latencia.items() if v is not None})

W0924 17:52:38.693000 15568 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


params_m         5.002373
size_mb_fp32    19.082539
gflops           4.000512
gmacs            2.000256
dtype: float64


,gpu,cpu
device,NVIDIA GeForce RTX 5060 Laptop GPU,cpu
batch_size,1,1
cpu_threads,1,1
n_runs,100,100
median_ms,10.40125,106.55155
p90_ms,11.57699,111.80461
mean_ms,10.475652,99.672589


## 14. Guardar métricas

Todo lo que se reporta de esta corrida en un solo archivo, `metrics.json`, junto a la receta de entrenamiento y las versiones de software. Es la entrada de la tabla comparativa con ViT.

In [19]:
import json

metrics = {
    "model_key": "mobilevit",
    "checkpoint": CHECKPOINT,
    "receta": {
        "epochs_max": EPOCHS,
        "epochs_entrenadas": trainer.state.epoch,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup": WARMUP,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "seed": int(config.SEED),
    },
    "tiempo_entrenamiento_s": elapsed_train,
    "best_val_f1_macro": trainer.state.best_metric,
    "test": test_metrics,
    "benchmark_por_tercil": benchmark.to_dict(orient="index"),
    "costo": costo,
    "latencia": latencia,
    "entorno": {
        "torch": torch.__version__,
        "transformers": __import__("transformers").__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
}
(RESULTS_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Guardado en", RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(" -", p.name)

Guardado en C:\Users\Antonella\Documents\Facultad\Maestría en IA\VC III\Repo proyecto integrador\ViT-for-101-food-app\reports\results\mobilevit
 - benchmark_por_tercil.csv
 - confusiones_test.csv
 - metrics.json
 - predictions_test.csv
 - report_por_clase_test.csv


## Resumen

- `train` completo para entrenar; `validation` para early stopping y selección de checkpoint; `test` una sola vez al final.
- La métrica que contesta la pregunta del proyecto es la del **subset del benchmark por tercil** (sección 10), no el accuracy global.
- El costo (sección 13) se mide con el mismo código que usa ViT, en el mismo equipo.

Archivos en `reports/results/mobilevit/` (livianos, pensados para versionarse):

| Archivo | Contenido |
|---|---|
| `predictions_test.csv` | Una fila por imagen de test: clase real, predicha, acierto top-1 y top-5 |
| `benchmark_por_tercil.csv` | Métricas sobre el subset del benchmark, global y por tercil |
| `report_por_clase_test.csv` | Precision, recall y F1 por clase |
| `confusiones_test.csv` | Pares (real → predicha) ordenados por cantidad de errores |
| `metrics.json` | Todo lo anterior resumido, más receta, tiempos, costo y versiones |

**En Colab** con `USE_DRIVE = True`, estos archivos quedan en `ceia-vpc3/results/mobilevit/` de tu Drive: para versionarlos, descargalos a `reports/results/mobilevit/` del checkout local y commitealos (mismo circuito que los artefactos del EDA).

Los checkpoints quedan en `models/mobilevit/full/` y **no** se versionan: se regeneran corriendo esta notebook.